In [13]:
import torch
from torch import nn,optim
from torch.nn import functional as F

#导入torchvision读取图片
from torchvision import datasets,transforms
#MNIST可通过pytorchapi直接接入
from torchvision.datasets import MNIST

#导入dataloader加载数据
from torch.utils.data import DataLoader

#可视化
from tqdm import tqdm

In [23]:
#超参数
batch_size=256
lr=1e-4
epochs=20

In [24]:
device='cuda:0'
#Resnet模型定义
#残差块定义：
class Block(nn.Module):
    def __init__(self,res_in_channels,res_out_channels,stride1=1,stride2=1):
        super().__init__()
        self.res=nn.Sequential(
            nn.Conv2d(in_channels=res_in_channels,out_channels=res_out_channels,kernel_size=3,padding=1,stride=stride1),
            nn.BatchNorm2d(res_out_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels=res_out_channels,out_channels=res_out_channels,kernel_size=3,padding=1,stride=stride2),
            nn.BatchNorm2d(res_out_channels),
        )
    def forward(self,x):
        x=self.res(x)
        
        return F.relu(x)


class Resnet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv0=nn.Sequential(
            nn.Conv2d(1,64,kernel_size=7,stride=2,padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2,padding=1),
        )
        self.resnet1=Block(64,64)
        self.resnet2=Block(64,64)
        self.resnet3=Block(64,128,stride1=2,stride2=1)
        self.resnet4=Block(128,128)
        self.resnet5=Block(128,256,stride1=2,stride2=1)
        self.resnet6=Block(256,256)
        self.resnet7=Block(256,512,stride1=2,stride2=1)
        self.resnet8=Block(512,512)
        self.globalpool=nn.AdaptiveAvgPool2d((1, 1))
        self.fc=nn.Linear(512,1000)

        self.net1=nn.Conv2d(64,128,kernel_size=1,stride=2,padding=0)
        self.net2=nn.Conv2d(128,256,kernel_size=1,stride=2,padding=0)
        self.net3=nn.Conv2d(256,512,kernel_size=1,stride=2,padding=0)
    def forward(self,x):
        x0=self.conv0(x)
        out1=x0
        x1=self.resnet1(x0)
        x1=x1+out1
        out2=x1
        x2=self.resnet2(x1)
        x2=x2+out2
        out3=x2
        x3=self.resnet3(x2)
        #out3需要1x1卷积过滤
        
        out3=self.net1(out3)

        x3=x3+out3
        out4=x3
        x4=self.resnet4(x3)
        x4=x4+out4
        out5=x4
        x5=self.resnet5(x4)
        #out5需要过滤
        
        out5=self.net2(out5)
        x5=x5+out5
        out6=x5
        x6=self.resnet6(x5)
        x6=x6+out6


        out7=x6
        x7=self.resnet7(x6)
        #out7需要过滤
        
        out7=self.net3(out7)
        x7=x7+out7
        out8=x7
        x8=self.resnet8(x7)
        x8=x8+out8

        y=self.globalpool(x8)
        y=torch.flatten(y,start_dim=1)
        y=self.fc(y)
        return y
model=Resnet()    
model.to(device) #记得迁移到gpu上去
                

Resnet(
  (conv0): Sequential(
    (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (resnet1): Block(
    (res): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (resnet2): Block(
    (res): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(64, 

# 数据加载：不熟悉

In [25]:
#dataset/dataloader
#数据加载：数据预处理
transform = transforms.Compose([
    transforms.Pad(2),#默认28x28的尺寸填充为32x32尺寸
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5],std = [0.5])

    ]) #预处理函数，在下载时作为参数
path='./data/'

#只是下载，返回dataset对象
train_dataset=MNIST(path,train=True,transform=transform,download=True)  #下载，并执行预处理
test_dataset=MNIST(path,train=False,transform=transform)

#分割为一份一份batch,训练时的batch_x、batch_y来源
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)#shuffle是为了打乱顺序
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False)



#如果自定义dataloader，目标是返回（x-y）的tuple
'''
class MyDataLoader:
    def __init__(self, dataset, batch_size, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = list(range(len(dataset)))
    
    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.indices)
        self.current = 0
        return self
    
    def __next__(self):
        if self.current >= len(self.indices):
            raise StopIteration
        batch_indices = self.indices[self.current:self.current + self.batch_size]
        batch = [self.dataset[i] for i in batch_indices]
        # 假设 dataset[i] 返回 (x, y)
        xs = torch.stack([item[0] for item in batch])
        ys = torch.tensor([item[1] for item in batch])
        self.current += self.batch_size
        return xs, ys
'''

'\nclass MyDataLoader:\n    def __init__(self, dataset, batch_size, shuffle=True):\n        self.dataset = dataset\n        self.batch_size = batch_size\n        self.shuffle = shuffle\n        self.indices = list(range(len(dataset)))\n\n    def __iter__(self):\n        if self.shuffle:\n            random.shuffle(self.indices)\n        self.current = 0\n        return self\n\n    def __next__(self):\n        if self.current >= len(self.indices):\n            raise StopIteration\n        batch_indices = self.indices[self.current:self.current + self.batch_size]\n        batch = [self.dataset[i] for i in batch_indices]\n        # 假设 dataset[i] 返回 (x, y)\n        xs = torch.stack([item[0] for item in batch])\n        ys = torch.tensor([item[1] for item in batch])\n        self.current += self.batch_size\n        return xs, ys\n'

In [26]:
#优化器、损失函数
#优化函数定义
criterion=nn.CrossEntropyLoss()
optimizer=optim.AdamW(model.parameters(),lr=lr) #必须要在模型之后

In [27]:
#训练函数
#训练函数
for epoch in range(epochs):
    epoch_loss=0
    for batch_x,batch_y in tqdm(train_loader):
        #一定要保证模型与数据都迁移了
        batch_x = batch_x.to(device)  
        batch_y = batch_y.to(device)
        y_pred=model(batch_x)
        loss=criterion(y_pred,batch_y)
        epoch_loss+=loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if epoch%10==0:
        print(f'epoch:{epoch},loss:{epoch_loss/batch_size}')


100%|██████████| 938/938 [00:37<00:00, 24.75it/s]


epoch:0,loss:0.6013334393501282


 15%|█▌        | 142/938 [00:05<00:32, 24.26it/s]


KeyboardInterrupt: 

# 测试模型：不熟悉

In [11]:
torch.save(model.state_dict(),'lenet_mnist.params')



# 1. 加载模型
test_model = Resnet()
test_model.load_state_dict(torch.load('lenet_mnist.params',weights_only=False))
test_model.to(device)  # 模型移到 GPU
test_model.eval()       # 切换到评估模式

# 2. 测试
correct = 0
total = 0

with torch.no_grad():  # 测试时不需要计算梯度     train的外侧是for循环，test则是关闭梯度计算：内部几乎一致
    for batch_x, batch_y in tqdm(test_loader):
        #几乎与train部分一致，保证都迁移到gpu
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        output = test_model(batch_x)#output是一个单维度向量，train的时候的y_pred

        # 取概率最大的类别作为预测结果
        _, predicted = torch.max(output, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = 100 * correct / total
print(f'测试集准确率: {accuracy:.2f}%')#测试

NameError: name 'model' is not defined